In [1]:
import sys
print(sys.executable)

C:\Users\dell\smart-doc-qa\venv\Scripts\python.exe


In [1]:
from pypdf import PdfReader

In [2]:
#extracting data from the pages 

reader=PdfReader("data/TOE - Naxrita.pdf")

text = ""
pages=[]
for page_num,page in enumerate(reader.pages, start=1):
    page_text=page.extract_text()
    if page_text:
        text+=page_text
        pages.append({
            "page_num":page_num,
            "page_text":page_text
        })
print(text[:500]) 
print(len(text))
print(pages[0])

Naxrita Solutions Pvt Ltd, India 
Employee ID Number 
TERMS OF EMPLOYMENT 
 
Your employment for Naxrita Solutions Private Limited (“Company" or “Naxrita") will be 
governed by Company's policies, as modified, from time to time and at the Company's 
sole discretion, upon notice to you. The terms and conditions contained herein ("Terms 
of Employment™") must be read in conjunction with Company policies. Any policy 
infraction will amount  to breach of your terms  of employment and may  lead to 
t
37824
{'page_num': 1, 'page_text': 'Naxrita Solutions Pvt Ltd, India \nEmployee ID Number \nTERMS OF EMPLOYMENT \n \nYour employment for Naxrita Solutions Private Limited (“Company" or “Naxrita") will be \ngoverned by Company\'s policies, as modified, from time to time and at the Company\'s \nsole discretion, upon notice to you. The terms and conditions contained herein ("Terms \nof Employment™") must be read in conjunction with Company policies. Any policy \ninfraction will amount  to breach o

In [3]:
#Chunking function

def chunk_text(pages, chunk_size=1000, chunk_overlap=200):

    chunks = []
    chunk_id = 1
    step = chunk_size - chunk_overlap

    # Process one page at a time
    for page in pages:

        page_num = page["page_num"]
        page_text = page["page_text"]

        start = 0

        while start < len(page_text):

            chunk = page_text[start:start + chunk_size]

            chunks.append({
                "page_num": page_num,
                "chunk_id": chunk_id,
                "text": chunk
            })

            chunk_id += 1
            start += step

    return chunks

In [4]:
chunks = chunk_text(pages)
print(f"Total chunks: {len(chunks)}")
print(len(chunks[-1]["text"]))

Total chunks: 50
606


In [5]:
#Embeddings
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = model.encode(texts)
print(type(chunk_embeddings))
print(chunk_embeddings.shape)

C:\Users\dell\smart-doc-qa\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2513.81it/s]


<class 'numpy.ndarray'>
(50, 384)


In [6]:
for chunk, embedding in zip(chunks, chunk_embeddings):
    chunk["embedding"] = embedding
print(chunks[0])

{'page_num': 1, 'chunk_id': 1, 'text': 'Naxrita Solutions Pvt Ltd, India \nEmployee ID Number \nTERMS OF EMPLOYMENT \n \nYour employment for Naxrita Solutions Private Limited (“Company" or “Naxrita") will be \ngoverned by Company\'s policies, as modified, from time to time and at the Company\'s \nsole discretion, upon notice to you. The terms and conditions contained herein ("Terms \nof Employment™") must be read in conjunction with Company policies. Any policy \ninfraction will amount  to breach of your terms  of employment and may  lead to \ntermination of your services. These Terms of Employment and policies shall be subject \nto modifications, from time to time, upon notice to you. \n \n1. Probation \n1.1 If your management level is 5 to 11 \n \n1 .1.1   You shall be on probation for a period of one hundred and eighty ( 180) calendar days from the effective start date of your \nemployment with Company. Company may, in its sole discretion, at anytime extend this period of probation 

In [7]:
import numpy as np

query = "What is the probation period?"
query_embedding = model.encode(query)

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

In [8]:
#Comparing the query embedding with all the chunks
scores=[]

for idx in range(len(chunks)):
    score = cosine_similarity(query_embedding, chunks[idx]["embedding"])
    scores.append(score)
print(scores[1])

0.61809117


In [9]:
#Finding out the best score and chunk manually
high_score=scores[0]
best_idx=chunks[0]["chunk_id"]

for idx in range(len(scores)):
    if(scores[idx]>high_score):
        high_score=scores[idx]
        best_idx=chunks[idx]["chunk_id"]

print(high_score, best_idx)

0.6483511 4


In [10]:
#Using method
top_indices=np.argsort(scores)[::-1][:3]
top_chunks = []

for idx in top_indices:
    top_chunks.append(chunks[idx]["text"])

context="\n\n".join(top_chunks)
print(context)

e (365) days 
after your effective start date, unless the probation period is extended up to an additional sixty (60) day period, in which case, 
the probation period shall expire after a maximum of four hundred twenty five (425) days after your effective start date. 
 
1.2.3 Notwithstanding anything contained herein, during your probation period, Company may terminate your employment upon 
thirty (30) calendar days notice to you or by paying your monthly gross salary  in lieu of giving such notice, with or without 
cause, and with or without stating any reasons whatsoever. 
 
1.3           If you desire to terminate your employment during the probation period, you shall provide 
      Company thirty (30) calendar days prior written notice with reasons for such termination 
 
2. Employee screening 
 
2,1             You acknowledge and agree that Company has offered you employment based on the specific information and records furnished by        
you or on your behalf. You will provide

In [11]:
prompt = f"""
You are a helpful assistant.

Answer the user's question using only the information provided in the context below.

If the answer is not present in the context, reply:
"Information not found in the document."

Context:
{context}

Question:
{query}

Answer:
"""

In [16]:
import ollama
response = ollama.chat(
    model="llama3.2",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response["message"]["content"]

print(answer)

The probation period is 365 days from the effective start date of your employment with the Company. However, it can be extended up to an additional 60 days at the Company's sole discretion upon notice to you.
e (365) days 
after your effective start date, unless the probation period is extended up to an additional sixty (60) day period, in which case, 
the probation period shall expire after a maximum of four hundred twenty five (425) days after your effective start date. 
 
1.2.3 Notwithstanding anything contained herein, during your probation period, Company may terminate your employment upon 
thirty (30) calendar days notice to you or by paying your monthly gross salary  in lieu of giving such notice, with or without 
cause, and with or without stating any reasons whatsoever. 
 
1.3           If you desire to terminate your employment during the probation period, you shall provide 
      Company thirty (30) calendar days prior written notice with reasons for such termination 
 
2. E